# Laxman AI Avatar Studio — MuseTalk v1.5 Worker

Permanent notebook in GitHub; private media and model weights remain in Google Drive. Run **BOOTSTRAP** after a fresh Colab runtime, then **RUN QUEUE** for one job.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT=Path('/content/drive/MyDrive/Laxman AI Avatar Studio')
for p in ['avatars','voices','models','outputs','temp','logs','jobs/queued','jobs/processing','jobs/completed','jobs/failed']:
    (ROOT/p).mkdir(parents=True,exist_ok=True)
print(ROOT)

In [ ]:
# BOOTSTRAP — safe to rerun after a runtime reset
%cd /content
!git clone https://github.com/LaxmanNepal/AI-Avatar-Studio.git /content/AI-Avatar-Studio 2>/dev/null || true
!python3.10 /content/AI-Avatar-Studio/backend/colab_bootstrap.py

In [ ]:
# PHASE 11 — CREATE A REAL TEST JOB FROM YOUR EXISTING DRIVE FILES
from pathlib import Path
import json, uuid
avatar=ROOT/'temp/laxman_avatar_test.mp4'
audio=ROOT/'temp/laxman_voice_musetalk.wav'
assert avatar.exists(), f'Missing avatar: {avatar}'
assert audio.exists(), f'Missing audio: {audio}'
job_id='laxman-test-'+uuid.uuid4().hex[:8]
job={'schema_version':1,'id':job_id,'avatar_path':str(avatar),'audio_path':str(audio),'batch_size':4}
job_file=ROOT/'jobs/queued'/f'{job_id}.json'
job_file.write_text(json.dumps(job,indent=2))
print('TEST JOB READY')
print(job_file)
print(json.dumps(job,indent=2))

In [ ]:
# RUN QUEUE — processes one Drive job
!python /content/AI-Avatar-Studio/backend/run_worker.py

In [ ]:
# QUICK HEALTH CHECK
from pathlib import Path
import json
for p in ['models/musetalkV15/unet.pth','models/musetalkV15/musetalk.json','models/sd-vae/config.json','models/whisper/config.json']:
    x=ROOT/p; print('OK' if x.exists() else 'MISSING', x)
print('Queued:',len(list((ROOT/'jobs/queued').glob('*.json'))))
print('Outputs:',len(list((ROOT/'outputs').glob('*.mp4'))))

## Job example

Create a JSON file in `jobs/queued/`, using absolute Drive paths. Example:

```json
{
  "schema_version": 1,
  "id": "test-001",
  "avatar_path": "/content/drive/MyDrive/Laxman AI Avatar Studio/temp/laxman_avatar_test.mp4",
  "audio_path": "/content/drive/MyDrive/Laxman AI Avatar Studio/temp/laxman_voice_musetalk.wav",
  "batch_size": 4
}
```

The worker keeps only code/config in GitHub and never uploads private media to the repository.

In [ ]:
# PHASE 12 — INSPECT THE RESULT
from pathlib import Path
outputs=sorted((ROOT/'outputs').glob('*.mp4'),key=lambda p:p.stat().st_mtime,reverse=True)
print('Generated MP4 files:',len(outputs))
for p in outputs[:10]: print(p.name, round(p.stat().st_size/1024/1024,2),'MB')